# 05 Create SFINCS Scenarios

> Requires: wave-coupled base model, event catalog, rainfall, waves, surge, soil moisture

In [ ]:
import sys
from pathlib import Path

location_root = Path("..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

import json
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

# model, scenario, storage, and catalog paths.
from sfincs_runs.notebook import load_runtime
# materializes catalog rows into SFINCS scenario folders.
from sfincs_runs.scenarios.create_events import parse_args, build_scenarios

runtime = load_runtime(location_root, wave=True, create_base_model_dir=False)
config = runtime.config
paths = runtime.paths
# None means build every Event Catalog row, including appended historical-tail references.
scenario_limit = None
workers_per_cpu_node = 32
array_chunks = 16
max_concurrent_cpu_nodes = 4
event_catalog_path = runtime.catalog_dir / "event_catalog.csv"
expected_scenario_count = len(pd.read_csv(event_catalog_path, usecols=["event_id"])) if event_catalog_path.exists() else pd.NA

display(pd.Series({
    "location": runtime.location_name,
    "base_model_root": str(runtime.base_model),
    "scenarios_root": str(runtime.scenarios_root),
    "storage_root": str(runtime.storage_root),
    "run_root": str(runtime.run_root),
    "scenario_limit": scenario_limit,
    "expected_scenario_count": expected_scenario_count,
}, name="create_scenarios_paths"))


## Rerun Control


In [ ]:
rerun = False


## Required Inputs

The scenario builder consumes the wave-coupled base model and Event Catalog outputs. Missing inputs here mean the cluster job is not ready to submit.


In [ ]:
wave_base = runtime.base_model
required = {
    "wave_base_model": wave_base,
    "base_sfincs_inp": wave_base / "sfincs.inp",
    "base_boundary": wave_base / "sfincs.bnd",
    "base_snapwave_boundary": wave_base / "snapwave.bnd",
    "base_runup_gauges": wave_base / "sfincs.rug",
    "event_catalog": paths["design_outputs_root"] / "catalog" / "event_catalog.csv",
    "event_catalog_audit": paths["design_outputs_root"] / "catalog" / "event_catalog_audit.json",
    "surge_members": paths["design_outputs_root"] / "events" / "surge_event_members.nc",
    "data_catalog": paths["data_catalog"],
}
status = pd.DataFrame([
    {"artifact": key, "path": str(path), "exists": Path(path).exists()}
    for key, path in required.items()
])
display(status)
missing = status.loc[~status["exists"], "artifact"].tolist()
if missing:
    raise FileNotFoundError("Missing required scenario-build inputs: " + ", ".join(missing))


## Build Full Wave-Coupled Scenario Folders

In [ ]:
builder_argv = [
    "--config", str(paths["location_config_path"]),
    "--include-waves",
    "--include-precip",
    "--zsini-mode", "boundary_t0",
]
if scenario_limit is not None:
    builder_argv.extend(["--limit", str(scenario_limit)])
builder_argv.append("--force" if rerun else "--resume")

print("python -m sfincs_runs build_scenarios " + " ".join(builder_argv))
args = parse_args(builder_argv)
report = build_scenarios(args)
display(report[[
    "event_id", "run_root", "run_duration_hours", "forcing_variable",
    "expected_has_waves", "expected_has_precip", "netamprfile",
    "rainfall_window_alignment", "expected_zsini_m",
]].head())


## Scenario Folder Audit

In [ ]:
scenario_root = paths["scenarios_root"]
scenario_catalog = pd.read_csv(scenario_root / "scenario_catalog.csv")
# Audit the staged run roots written by the builder; event IDs may be evt_* or design_* depending on the catalog route.
event_dirs = [Path(path) for path in scenario_catalog["run_root"]]
required_event_files = [
    "sfincs.inp", "sfincs.bnd", "sfincs.bzs", "forcing_manifest.json",
    "snapwave.bnd", "snapwave.bhs", "snapwave.btp", "snapwave.bwd", "snapwave.bds",
    "aorc_precip_for_sfincs.nc", "sfincs_netampr.nc", "sfincs.smax", "sfincs.seff", "sfincs.ks",
]

def netampr_status(event_dir, manifest):
    netampr = event_dir / str(manifest.get("netamprfile") or "sfincs_netampr.nc")
    if not netampr.exists():
        return "missing"
    run_start = manifest.get("run_start")
    run_stop = manifest.get("run_stop")
    if not run_start or not run_stop:
        return "missing_run_window"
    try:
        with xr.open_dataset(netampr) as ds:
            if "time" not in ds:
                return "missing_time_coord"
            times = pd.to_datetime(ds["time"].values)
            if len(times) == 0:
                return "empty_time_coord"
            if times[0] != pd.Timestamp(run_start) or times[-1] != pd.Timestamp(run_stop):
                return f"stale_time:{times[0]}..{times[-1]}"
            finite_any = False
            for name in ds.data_vars:
                values = ds[name].values
                if np.issubdtype(values.dtype, np.number) and np.isfinite(values).any():
                    finite_any = True
                    break
            return "ok" if finite_any else "all_nan"
    except Exception as exc:
        return f"error:{type(exc).__name__}"

audit_event_dirs = event_dirs if scenario_limit is None else event_dirs[: min(len(event_dirs), scenario_limit)]
expected_folder_count = expected_scenario_count if scenario_limit is None else scenario_limit
audit_rows = []
for event_dir in audit_event_dirs:
    missing = [name for name in required_event_files if not (event_dir / name).exists()]
    manifest = json.loads((event_dir / "forcing_manifest.json").read_text(encoding="utf-8")) if (event_dir / "forcing_manifest.json").exists() else {}
    audit_rows.append({
        "event_id": event_dir.name,
        "missing_files": ", ".join(missing),
        "netampr_status": netampr_status(event_dir, manifest),
        "run_duration_hours": manifest.get("run_duration_hours"),
        "forcing_variable": manifest.get("forcing_variable"),
        "waves": manifest.get("expected_has_waves"),
        "precip": manifest.get("expected_has_precip"),
    })
audit_frame = pd.DataFrame(audit_rows)
missing_count = int(audit_frame["missing_files"].astype(bool).sum()) if len(audit_frame) else 0
bad_netampr_count = int((audit_frame["netampr_status"] != "ok").sum()) if len(audit_frame) else 0
display(pd.Series({
    "scenario_root": str(scenario_root),
    "event_folders": len(event_dirs),
    "expected_folders": expected_folder_count,
    "audited_folders": len(audit_frame),
    "folders_with_missing_files": missing_count,
    "folders_with_bad_netampr": bad_netampr_count,
}, name="scenario_folder_audit"))
display(audit_frame.head(12))
if pd.notna(expected_folder_count) and len(event_dirs) < int(expected_folder_count):
    raise RuntimeError(f"Expected at least {int(expected_folder_count)} scenario folders, found {len(event_dirs)}")
if missing_count:
    bad = audit_frame[audit_frame["missing_files"].astype(bool)].head(5)
    raise RuntimeError("Scenario folders missing required forcing files:\n" + bad.to_string(index=False))
if bad_netampr_count:
    bad = audit_frame[audit_frame["netampr_status"] != "ok"].head(5)
    raise RuntimeError("Scenario folders have invalid precipitation NetCDFs:\n" + bad.to_string(index=False))


## Cluster Launch Plan

The Slurm script runs prepared scenario folders and keeps completed outputs compact: `sfincs_map.nc`, `sfincs_his.nc`, logs, manifest, and small boundary/structure receipts.

In [ ]:
cluster_debug_event_id = pd.read_csv(paths["scenarios_root"] / "scenario_catalog.csv")["event_id"].iloc[0]
limit_arg = "" if scenario_limit is None else f" --limit {scenario_limit}"
rerun_flag = int(rerun)
scenario_build_mode = "--force" if rerun else "--resume"

commands = [
    f"python -m sfincs_runs build_scenarios --config {paths['location_config_path']}{limit_arg} --include-waves --include-precip --zsini-mode boundary_t0 {scenario_build_mode}",
    "mkdir -p runs runs/debug",
    f"sbatch --array=0-0 --export=ALL,EVENT_IDS={cluster_debug_event_id},WORKERS=1,FORCE_RERUN={rerun_flag},KEEP_STAGE=1,OMP_STACKSIZE=512M cluster/run_sfincs_dsai_wave_coupled.slurm",
    f"sbatch --array=0-{array_chunks - 1}%{max_concurrent_cpu_nodes} --export=ALL,WORKERS={workers_per_cpu_node},FORCE_RERUN={rerun_flag},OMP_STACKSIZE=512M cluster/run_sfincs_dsai_wave_coupled.slurm",
]
for command in commands:
    print(command)

batch_plan = {
    "location": paths["location_name"],
    "config_yaml": str(paths["location_config_path"]),
    "base_model_root": str(wave_base),
    "scenarios_root": str(paths["scenarios_root"]),
    "storage_root": str(paths["storage_root"]),
    "run_root": str(paths["run_root"]),
    "scenario_limit": scenario_limit,
    "expected_scenario_count": expected_scenario_count,
    "workers_per_cpu_node": workers_per_cpu_node,
    "array_chunks": array_chunks,
    "max_concurrent_cpu_nodes": max_concurrent_cpu_nodes,
    "rerun": rerun,
    "commands": commands,
    "slurm_scripts": [str(repo_root / "cluster" / "run_sfincs_dsai_wave_coupled.slurm")],
    "slurm_log_dir": str(repo_root / "runs"),
    "debug_stage_dir": str(repo_root / "runs" / "debug"),
}
paths["run_root"].mkdir(parents=True, exist_ok=True)
plan_path = paths["run_root"] / "create_scenarios_cluster_plan.json"
plan_path.write_text(json.dumps(batch_plan, indent=2), encoding="utf-8")
display(pd.Series({"plan": str(plan_path), "slurm_script_count": len(batch_plan["slurm_scripts"])}))
